# Potentials and Forces

In this notebook, we explore the different potentials and forces used in our molecular dynamics (MD) simulation.

---

### Outline:
1. **Lennard-Jones Potential and Force**, as a function of distance for each pair.
2. **Modified Lennard-Jones Potential and Force**, incorporating $(\Delta \phi)$ for each pair.

---

## 1. Lennard-Jones Potential and Force

For the `particleDot` datatype, we use the ***Lennard-Jones (LJ)*** potential as a model for pairwise interactions, as the ***van der Waals*** forces.  
This simplification is appropriate since the MD simulation focuses on ***off-lattice*** particle movements. Degrees of freedom will later be accounted for by modifying the potential.

The potential $U(r)$ is given by:
$$
U_{LJ}(r) = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]
$$
- $r$: Distance between two particles
- $\epsilon$: Depth of the potential well, representing interaction strength
- $\sigma$: Distance at which the potential is zero.

The corresponding force $F(r)$ is obtained as the negative gradient of the potential:

$$
F_{LJ}(r) = -\frac{dU(r)}{dr} = 48\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - 0.5\left(\frac{\sigma}{r}\right)^{6} \right] \frac{1}{r}
$$

This describes the attractive and repulsive forces acting between particles as a function of distance $r$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

SIGMA = 1.0
EPSILON = 1.0

def lj_potential(r, sigma=SIGMA, epsilon=EPSILON):
    if np.any(r <= 0):
        raise ValueError("Distance r must be greater than zero.")
    return 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

def lj_force(r, sigma=SIGMA, epsilon=EPSILON):
    if np.any(r <= 0):
        raise ValueError("Distance r must be greater than zero.")
    return 48 * epsilon * (sigma**12 / r**13 - 0.5 * sigma**6 / r**7)

def plot_lj_potential_force(sigma=SIGMA, epsilon=EPSILON, LJsigma_epsilon1=False):
    """Plot Lennard-Jones Potential and Force."""
    r = np.linspace(0.9 * sigma, 3 * sigma, 400)
    potential = lj_potential(r, sigma, epsilon)
    force = lj_force(r, sigma, epsilon)

    fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
    
    # Plot Potential
    if LJsigma_epsilon1:
        ax[0].plot(r, lj_potential(r, 1, 1), label='Potential (σ=1, ε=1)', color='green', linewidth=2)
    ax[0].plot(r, potential, label='Potential', color='#1f77b4', linewidth=2)
    ax[0].axvline(sigma, color='gray', linestyle='--', label=r'$r = \sigma$')
    ax[0].axhline(0, color='gray', linestyle='-')
    ax[0].axhline(-epsilon, color='gray', linestyle='--', label=r'$U = -\epsilon$')
    ax[0].set_title('Lennard-Jones Potential', fontsize=16, fontweight='bold')
    ax[0].set_xlabel('Distance (r)', fontsize=14)
    ax[0].set_ylabel('Potential Energy (U)', fontsize=14)
    ax[0].legend(fontsize=12)
    ax[0].grid(which='both', linestyle='--', alpha=0.7)
    
    # Plot Force
    if LJsigma_epsilon1:
        ax[1].plot(r, lj_force(r, 1, 1), label='Force (σ=1, ε=1)', color='green', linewidth=2)
    ax[1].plot(r, force, label='Force', color='#ff7f0e', linewidth=2)
    ax[1].axhline(0, color='gray', linestyle='--', label=r'$F = 0$')
    ax[1].set_title('Lennard-Jones Force', fontsize=16, fontweight='bold')
    ax[1].set_xlabel('Distance (r)', fontsize=14)
    ax[1].set_ylabel('Force (F)', fontsize=14)
    ax[1].legend(fontsize=12)
    ax[1].grid(which='both', linestyle='--', alpha=0.7)
    
    fig.tight_layout()
    plt.show()

def create_interactive_plot_LJ():
    """Create interactive plot for Lennard-Jones Potential and Force."""
    sigma_slider = ipywidgets.FloatSlider(value=SIGMA, min=0.1, max=3, step=0.1, description='σ:')
    epsilon_slider = ipywidgets.FloatSlider(value=EPSILON, min=0.1, max=3, step=0.1, description='ε:')
    LJsigma_epsilon1_checkbox = ipywidgets.Checkbox(value=False, description='show for σ=1, ε=1')
    
    return ipywidgets.interactive(
        plot_lj_potential_force,
        sigma=sigma_slider,
        epsilon=epsilon_slider,
        LJsigma_epsilon1=LJsigma_epsilon1_checkbox
    )

plot2D_LJ = create_interactive_plot_LJ()
display(plot2D_LJ)

## 2. Modified Lennard-Jones Potential and Force

To account for additional degrees of freedom, we introduce a modification to the Lennard-Jones potential based on $\Delta \phi$, which represents orientation-dependent interactions. This is particularly useful for simulating helical or anisotropic particles. We use this form of potential and force for the `ParticleOriented` data type, where each particle can have rotation ($\phi$) and angular velocity ($\omega$). Alongside mass, the moment of inertia is also defined as an additional property.

The modified potential $U_{mod}(r, \Delta \phi)$ is defined as:

$$
U_{mod}(r, \Delta \phi) = U_{LJ}(r) + A(r) \cos(n \Delta \phi)
$$

Here, $A(r)$ is a distance-dependent scaling factor, and $n \Delta \phi$ represents the orientation-based interaction term.

The force corresponding to the modified potential is:

$$
\vec{F}_{mod}(r, \Delta \phi) = -\frac{\partial U_{mod}(r, \Delta \phi)}{\partial r} \hat{a}_r - \frac{\partial U_{mod}(r, \Delta \phi)}{\partial \Delta \phi} \hat{a}_{\Delta \phi}
$$

Expanding the terms of $\vec{F}_{mod}$, we obtain **Radial Force ($F_{r}$)** and **Angular Force ($F_{\phi}$)** :
$$
F_{r} = F_{LJ} - \frac{\partial A(r)}{\partial r} \cos(n \Delta \phi)
$$
$$
F_{\phi} = n A(r) \sin(n \Delta \phi)
$$

Below, we visualize the potential and the corresponding forces,Using this format for additive term:
$$
A(r) \cos(n \Delta \phi) = \frac{a}{r^m} \cos(n \Delta \phi)
$$
using these control parameters:  
*$\phi$-order: n*  
*Angular Scale: a*  
*Angular Order: m*  

In [ ]:
SIGMA = 1.0
EPSILON = 1.0

def function_m(delta_phi, n, alpha):
    return np.cos(n* delta_phi + alpha)

def lj_mod_potential(r, delta_phi, phi_order, angular_scale, angular_order, sigma=SIGMA, epsilon=EPSILON, alpha=np.pi): 
    lj_term = lj_potential(r, sigma=SIGMA, epsilon=EPSILON)
    A = angular_scale / r**angular_order
    angular_term = A * function_m(delta_phi, phi_order, alpha)
    
    return lj_term + angular_term

def lj_mod_force(r, delta_phi, phi_order, angular_scale, angular_order, sigma=SIGMA, epsilon=EPSILON):
    lj_term = lj_force(r, sigma=SIGMA, epsilon=EPSILON)
    
    A = angular_scale / r**angular_order
    dA_dr = -angular_order * A / r
    
    radial_force = lj_term - dA_dr * np.cos(phi_order * delta_phi)
    angular_force = phi_order * A * np.sin(phi_order * delta_phi)
    
    return radial_force, angular_force


def plot2D_lj_mod_potential_force(phi_order=1, angular_scale=0.1, angular_order=12):
    r = np.linspace(0.9 * SIGMA, 3 * SIGMA, 400)
    delta_phi = np.linspace(-np.pi, np.pi, 400)
    
    potential = lj_mod_potential(r, delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)
    radial_force, angular_force = lj_mod_force(r, delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Potential plot
    ax[0].plot(r, potential, label='Potential', color='#1f77b4', linewidth=2)
    ax[0].axvline(SIGMA, color='gray', linestyle='--', label=r'$r = \sigma$')
    ax[0].axhline(0, color='gray', linestyle='-')
    ax[0].axhline(-EPSILON, color='gray', linestyle='--', label=r'$U = -\epsilon$')
    ax[0].set_title('Modified Lennard-Jones Potential', fontsize=16, fontweight='bold')
    ax[0].set_xlabel('Distance (r)', fontsize=14)
    ax[0].set_ylabel('Potential Energy (U)', fontsize=14)
    ax[0].tick_params(axis='both', labelsize=12)
    
    # Add minimum point
    min_potential_idx = np.argmin(potential)
    min_potential_r = r[min_potential_idx]
    ax[0].scatter(min_potential_r, potential[min_potential_idx], color='red', label=r'$U_{min}$')
    ax[0].legend(fontsize=12, loc='upper right')
    ax[0].grid(True, linestyle='--', alpha=0.7)
    
    # Force plot
    ax[1].plot(r, radial_force, label='Radial Force', color='#ff7f0e', linewidth=2)
    ax[1].plot(r, angular_force, label='Angular Force', color='#2ca02c', linewidth=2)
    ax[1].axhline(0, color='gray', linestyle='--')
    ax[1].set_title('Force Components', fontsize=16, fontweight='bold')
    ax[1].set_xlabel('Distance (r)', fontsize=14)
    ax[1].set_ylabel('Force (F)', fontsize=14)
    ax[1].tick_params(axis='both', labelsize=12)
    ax[1].legend(fontsize=12, loc='upper right')
    ax[1].grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

def plot_heatmap_lj_mod_potential(phi_order=1, angular_scale=0.1, angular_order=12):
    """Plot heatmap of potential and force."""
    L = 100
    r = np.linspace(0.9 * SIGMA, 1.3 * SIGMA, L)
    delta_phi = np.linspace(-np.pi, np.pi, L)
    R, Delta_phi = np.meshgrid(r, delta_phi)
    
    potential_values = lj_mod_potential(R, Delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)
    
    fig, axs = plt.subplots(figsize=(7, 7) ,dpi=100)
      
    c1 = axs.contourf(R, Delta_phi, potential_values, levels=100, cmap="viridis")
    fig.colorbar(c1, ax=axs, label='Potential $U_{mod}$')
    axs.set_title('Potential $U_{mod}(r, \Delta\phi)$')
    axs.set_xlabel('Distance $r$')
    axs.set_ylabel('Δφ (rad)')
    
    plt.tight_layout()
    plt.show()

def plot_heatmap_lj_mod_potential_polar(phi_order=1, angular_scale=0.1, angular_order=12):
    """Plot heatmap of potential and force in polar coordinates with a cleaner look."""
    L = 100
    r = np.linspace(0.9 * SIGMA, 1.3 * SIGMA, L)
    delta_phi = np.linspace(0, 2 * np.pi, L)
    R, Delta_phi = np.meshgrid(r, delta_phi)
    
    potential_values = lj_mod_potential(R, Delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)
    
    fig, axs = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(7, 7), dpi=100)
      
    # Plot the heatmap
    c1 = axs.contourf(Delta_phi, R, potential_values, levels=100, cmap="viridis")
    fig.colorbar(c1, ax=axs, label='Potential $U_{mod}$')
    
    # Customize the polar plot
    axs.set_title('Potential $U_{mod}(r, \Delta\phi)$', pad=20)
    axs.set_xlabel('Δφ (rad)', labelpad=20)
    axs.set_ylabel('')
    
    axs.grid(False)  # Disable the grid
    axs.set_yticklabels([])  # Remove radial tick labels
    
    plt.tight_layout()
    plt.show()
    
def plot_heatmap_lj_mod_force(phi_order=1, angular_scale=0.1, angular_order=12):
    """Plot heatmap of potential and force."""
    L = 100
    r = np.linspace(0.9 * SIGMA, 1.3 * SIGMA, L)
    delta_phi = np.linspace(-np.pi, np.pi, L)
    R, Delta_phi = np.meshgrid(r, delta_phi)
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 7), dpi=100)    
    
    radial_force_values = lj_mod_force(R, Delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)[0]
    c2 = axs[0].contourf(R, Delta_phi, radial_force_values, levels=100, cmap="coolwarm")
    fig.colorbar(c2, ax=axs[0], label='Radial Force $F_r$')
    axs[0].set_title('Radial Force $F_r(r, \Delta\phi)$')
    axs[0].set_xlabel('Distance $r$')
    axs[0].set_ylabel('Δφ (rad)')
    
    angular_force_values = lj_mod_force(R, Delta_phi, phi_order, angular_scale, angular_order, SIGMA, EPSILON)[1]
    c3 = axs[1].contourf(R, Delta_phi, angular_force_values, levels=100, cmap="plasma")
    fig.colorbar(c3, ax=axs[1], label='Angular Force $F_\phi$')
    axs[1].set_title('Angular Force $F_\phi(r, \Delta\phi)$')
    axs[1].set_xlabel('Distance $r$')
    axs[1].set_ylabel('Δφ (rad)')

    plt.tight_layout()
    plt.show()


def create_interactive_plots():
    """Create interactive plots with widgets."""
    phi_order_slider = ipywidgets.IntSlider(value=2, min=1, max=10, step=1, description='φ order:')
    angular_scale_slider = ipywidgets.FloatSlider(value=1.6, min=0.1, max=5.0, step=0.1, description='Angular scale:')
    angular_order_slider = ipywidgets.IntSlider(value=12, min=1, max=20, step=1, description='Angular order(r^):')
    
    plot2D = ipywidgets.interactive(
        plot2D_lj_mod_potential_force,
        phi_order=phi_order_slider,
        angular_scale=angular_scale_slider,
        angular_order=angular_order_slider
    )
    
    heatmap_potential = ipywidgets.interactive(
        plot_heatmap_lj_mod_potential,
        phi_order=phi_order_slider,
        angular_scale=angular_scale_slider,
        angular_order=angular_order_slider
    )
    
    heatmap_potential_polar = ipywidgets.interactive(
        plot_heatmap_lj_mod_potential_polar,
        phi_order=phi_order_slider,
        angular_scale=angular_scale_slider,
        angular_order=angular_order_slider
    )
    
    heatmap_force = ipywidgets.interactive(
        plot_heatmap_lj_mod_force,
        phi_order=phi_order_slider,
        angular_scale=angular_scale_slider,
        angular_order=angular_order_slider
    )
        
    return plot2D, heatmap_potential, heatmap_potential_polar, heatmap_force

plot2D, heatmap_potential, heatmap_potential_polar, heatmap_force = create_interactive_plots()
display(plot2D, heatmap_potential, heatmap_potential_polar, heatmap_force)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

def function_m(delta_phi, n, alpha):
    return np.cos(n* delta_phi + alpha)

def plot_orientational_function(n, alpha):
    delta_phi = np.linspace(-np.pi, np.pi, 400)
    data = function_m(delta_phi, n, alpha)

    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(delta_phi, data, label='Potential', color='#1f77b4', linewidth=2)
    ax.set_title('orientational term', fontsize=16, fontweight='bold')
    ax.set_xlabel('$\Delta \phi$', fontsize=14)
    ax.set_ylabel('Potential Energy factor', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(which='both', linestyle='--', alpha=0.7)
    
    fig.tight_layout()
    plt.show()

def create_interactive_plot_or():
    n_slider = ipywidgets.FloatSlider(value=1, min=1, max=10, step=1, description='n:')
    alpha_slider = ipywidgets.FloatSlider(value=0, min=0, max=2*np.pi, step=np.pi/4, description='$\alpha$:')
    
    return ipywidgets.interactive(
        plot_orientational_function,
        n=n_slider,
        alpha=alpha_slider,
    )

plot2D_or = create_interactive_plot_or()
display(plot2D_or)